# DDLR host-galaxy matching with LSDB

Crossmatch an input transient catalog against a galaxy catalog (Rubin DP2 object or Legacy Survey DR10.1), compute DDLR for every candidate, and keep the smallest-DDLR galaxy as the host.

In [ ]:
# %pip install lsdb --upgrade
# %pip install nested-pandas --upgrade

In [ ]:
from pathlib import Path

import lsdb
import nested_pandas
import numpy as np
from dask.distributed import Client

from ddlr import RUBIN_SHAPE_COLS, LEGACY_SHAPE_COLS, add_ddlr, select_host

In [ ]:
lsdb.__version__, nested_pandas.__version__

## Config

In [ ]:
GALAXY_CATALOG = "rubin"  # "rubin" or "legacy"

# input transient catalog (HATS)
input_path = ''  # TODO: set path
INPUT_ID_COL = 'diaObjectId'
INPUT_COLUMNS = [INPUT_ID_COL]  # ra/dec are always loaded

object_path = '/astro/store/shire/hats/dash/hats/dp2_rc/object_collection'
ls_path = '/astro/store/shire/hats/catalogs/legacysurvey_dr10.1/'

SEARCH_RADIUS_ARCSEC = 30
N_NEIGHBORS = 10
DDLR_MAX = 4

SN_SUFFIX, GAL_SUFFIX = "_sn", "_gal"

outdir = Path('outputs/')

In [ ]:
# client = Client(n_workers=1, memory_limit="10 GiB", threads_per_worker=1)
client = Client(n_workers=20, memory_limit="24 GiB", threads_per_worker=1)
display(client)

## Load catalogs

In [ ]:
transients = lsdb.open_catalog(input_path, columns=INPUT_COLUMNS)
transients

In [ ]:
if GALAXY_CATALOG == "rubin":
    galaxy_columns = ["objectId", "refExtendedness", "refSizeExtendedness", *RUBIN_SHAPE_COLS.values()]
    objects = lsdb.open_catalog(object_path, columns=galaxy_columns)
    galaxies = objects.query("refExtendedness > 0.7 & refSizeExtendedness > 0.7")
elif GALAXY_CATALOG == "legacy":
    galaxy_columns = ["OBJID", "TYPE", "Z_PHOT_MEAN", "Z_PHOT_STD", "Z_SPEC", *LEGACY_SHAPE_COLS.values()]
    objects = lsdb.open_catalog(ls_path, columns=galaxy_columns)
    galaxies = objects.query("TYPE != 'PSF'")
else:
    raise ValueError(GALAXY_CATALOG)
galaxies.columns

## Candidate hosts
All galaxies within `SEARCH_RADIUS_ARCSEC` (up to `N_NEIGHBORS` per transient). lsdb appends the suffixes to the column names; check `candidates.columns` if the names below don't match.

In [ ]:
candidates = transients.crossmatch(galaxies, n_neighbors=N_NEIGHBORS, radius_arcsec=SEARCH_RADIUS_ARCSEC,
                                   suffixes=(SN_SUFFIX, GAL_SUFFIX))
candidates.columns

In [ ]:
sn_info = transients.hc_structure.catalog_info
gal_info = galaxies.hc_structure.catalog_info
cols = {
    "ra_sn": sn_info.ra_column + SN_SUFFIX,
    "dec_sn": sn_info.dec_column + SN_SUFFIX,
    "ra_gal": gal_info.ra_column + GAL_SUFFIX,
    "dec_gal": gal_info.dec_column + GAL_SUFFIX,
    "sep": "_dist_arcsec",
}
cols

## DDLR and host selection

In [ ]:
with_ddlr = candidates.map_partitions(add_ddlr, galaxy_catalog=GALAXY_CATALOG, cols=cols, gal_suffix=GAL_SUFFIX)

In [ ]:
hosts = with_ddlr.map_partitions(select_host, id_col=INPUT_ID_COL + SN_SUFFIX, ddlr_max=DDLR_MAX)

In [ ]:
# quick check on a small piece before the full run
# hosts.head()
# candidates.map_partitions(add_ddlr, galaxy_catalog=GALAXY_CATALOG, cols=cols, gal_suffix=GAL_SUFFIX,
#                           compute_single_partition=True, partition_index=0)

In [ ]:
hosts.write_catalog(outdir / f"ddlr_hosts_{GALAXY_CATALOG}", overwrite=True)

In [ ]:
client.close()